# Inverse 2-D Heat-Transfer PINN

**Author:** Ivan Lisovskyi 

**Goal:** reconstruct the temperature field in a 2-D plate and estimate the unknown thermal diffusivity $\alpha$ from sparse COMSOL sensor data.

**Method:** inverse Physics-Informed Neural Network (PINN). The network is trained on COMSOL sensor data, the heat equation, initial/boundary conditions, and a trainable positive parameter $\alpha$.

## 1. Imports and constants

Known values come from the COMSOL model. The reference value of $\alpha$ is kept only for final error calculation, not for the training loss.

In [ ]:
import os, math, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


Lx, Ly, T_MAX = 0.1, 0.05, 300.0      # plate size [m], final time [s]

# Temperature scale used for the normalised PINN output u=(T-T0)/dT
T0, dT = 293.15, 30.0                 # ambient temperature and heating amplitude [K]

# Left boundary fast time ramp and Gaussian shape along y
TAU = 0.5                             # heating ramp time constant [s]
SIGMA = Ly / 3.0                      # width of the heated spot [m]

# Material and boundary parameters:
RHO, CP = 7850.0, 470.0               # density [kg/m^3], heat capacity [J/(kg K)]
H_CONV, EPS = 10.0, 0.8               # convection coefficient and emissivity
SIGMA_SB = 5.670374419e-8             # Stefan-Boltzmann constant

# True alpha and initial alpha
ALPHA_TRUE = 1.0e-5
ALPHA_INIT = 1.5e-5                   # starting guess for the trainable alpha

# Training choices:
T_PDE_MIN = 0.5                       # avoid the hardest t=0 corner in the PDE points
NFF = 32                              # number of Fourier features per group
SIGMA_XY = 3.0                        # Fourier frequency scale for space
SIGMA_T = 2.5                         # Fourier frequency scale for time
EARLY_FRAC = 0.4                      # fraction of sampled points biased to early times
EARLY_POW = 3.0                       # strength of that early-time bias

# Device and output folder.
# MPS is Apple's GPU backend. If it is unavailable, the notebook uses CPU
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
OUT_DIR = "outputs_submission"
os.makedirs(OUT_DIR, exist_ok=True)

# Two-stage training: first fit the temperature field, then release alpha.
STAGE1 = 2000
STAGE2 = 6000
SEED = 0

# Loss weights and alpha learning rate.
W_DATA = 1.0
W_PDE = 1.0
W_BC = 0.05                           # small because the BC residual is normalised
ALPHA_LR = 3e-4                       # smaller than network LR to avoid alpha jumps

# Mini-batch sizes. These only control speed/memory, not the physical model.
DATA_BATCH = 2048                     # measured COMSOL points per epoch
PDE_BATCH = 2048                      # interior physics points per epoch
BC_BATCH = 512                        # boundary/initial points per epoch
VAL_BATCH = 8000                      # random held-out points used for RMSE estimate
LOG_EVERY = 200                       # save/print history every N epochs

def set_seed(seed):
    """Make random choices repeatable for this notebook run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

print("device:", DEVICE)


## 2. Load COMSOL sensor data

The CSV contains time in the first column and temperature traces in the remaining sensor columns. Sensor coordinates are read from the COMSOL column names.

In [ ]:
def parse_sensor_column(column_name):
    """Read one COMSOL sensor header and return its (x, y) position."""
    # Example header: "Temperature (K), Point: (0.0115, 0.005)".
    # Only the text after "Point:" is needed here.
    coordinate_text = column_name.split("Point:", 1)[1].strip()
    coordinate_text = coordinate_text.strip("()")
    x_text, y_text = coordinate_text.split(",")
    return float(x_text), float(y_text)


def load_comsol_sensor_data(csv_path="sensors_T.csv"):
    """Load the exported COMSOL sensor table.

    Returns:
    - time_values: shape [Nt]
    - sensor_x_positions, sensor_y_positions: shape [Ns]
    - temperature_values: shape [Nt, Ns], in Kelvin
    """
    # COMSOL puts four metadata rows before the actual table header
    sensor_table = pd.read_csv(csv_path, skiprows=4)

    # First column: simulation time in seconds
    time_values = sensor_table.iloc[:, 0].to_numpy(dtype=np.float32)

    # Remaining columns: one temperature trace per sensor point
    sensor_x_positions = []
    sensor_y_positions = []
    for column_name in sensor_table.columns[1:]:
        x_position, y_position = parse_sensor_column(column_name)
        sensor_x_positions.append(x_position)
        sensor_y_positions.append(y_position)

    # Matrix layout: rows are time steps, columns are sensor locations.
    temperature_values = sensor_table.iloc[:, 1:].to_numpy(dtype=np.float32)

    return (
        time_values,
        np.array(sensor_x_positions, dtype=np.float32),
        np.array(sensor_y_positions, dtype=np.float32),
        temperature_values,
    )

t_np, x_np, y_np, T_np = load_comsol_sensor_data()
print(f"timesteps={T_np.shape[0]}, sensors={T_np.shape[1]}, max rise={T_np.max()-T0:.2f} K")


### Figure: COMSOL sensor layout before training

This plot is made directly from the exported COMSOL sensor CSV. It shows where the virtual sensors are located on the rectangular plate before any PINN training is done.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))

# Draw the physical plate.
ax.add_patch(plt.Rectangle((0, 0), Lx, Ly, fill=False, lw=1.5, edgecolor="black"))

# The left edge is the heated boundary from the COMSOL setup.
# No label is added here because the slide only needs the sensor layout.
ax.plot([0, 0], [0, Ly], color="#d73027", lw=3.0)

# Each dot is one exported COMSOL temperature sensor.
ax.scatter(x_np, y_np, s=42, color="#2563eb", edgecolor="white", linewidth=0.6, zorder=3)

ax.set_xlim(-0.004, Lx + 0.004)
ax.set_ylim(-0.004, Ly + 0.004)
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("COMSOL virtual sensor locations")
ax.grid(alpha=0.18, lw=0.6)

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "sensor_plate_layout.png"), dpi=180)
plt.show()
plt.close(fig)


## 3. Neural network and trainable $\alpha$

The network predicts normalised temperature $u=(T-T_0)/\Delta T$. A small `tanh` MLP with Fourier features is used, and $\alpha$ is stored as `log_alpha` so it stays positive.

In [ ]:
def to_hat(x, y, t):
    """Convert physical coordinates to the normalised network input h=[xhat,yhat,that].

    h is just a short name for the input tensor passed into the neural network.
    """
    # x, y, t are torch tensors with the same length N.
    # Output shape is [N, 3], which is what the neural network expects.
    return torch.stack([x/Lx, y/Ly, t/T_MAX], dim=1)

class PINN(nn.Module):
    def __init__(self, width=64, depth=4, n_ff=NFF, sigma_xy=SIGMA_XY, sigma_t=SIGMA_T):
        """Fourier-feature MLP used as u_theta(x,y,t).

        n_ff means number of Fourier features. width/depth set the MLP size.
        """
        super().__init__()

        # B_xy and B_t are fixed random Fourier matrices.
        # They are buffers: saved with the model and moved to GPU/CPU, but not trained.
        self.register_buffer("B_xy", torch.randn(2, n_ff) * sigma_xy)   # maps [xhat,yhat]
        self.register_buffer("B_t",  torch.randn(1, n_ff) * sigma_t)    # maps [that]

        # Four groups enter the MLP: sin(xy), cos(xy), sin(t), cos(t).
        # in_dim means input dimension after the Fourier feature expansion.
        in_dim = 4 * n_ff
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width,width), nn.Tanh()]
        layers += [nn.Linear(width,1)]                                  # one output: normalised u
        self.net = nn.Sequential(*layers)

        # Store log(alpha), not alpha itself, so exp(log_alpha) is always positive.
        self.log_alpha = nn.Parameter(torch.tensor(math.log(ALPHA_INIT)))

    def features(self, h):
        """Build Fourier features from h with shape [N,3]."""
        xy = h[:, 0:2]                                                   # first two columns: xhat,yhat
        t = h[:, 2:3]                                                     # third column: that
        pxy = xy @ self.B_xy                                              # projected x-y coordinates
        pt  = t  @ self.B_t                                               # projected time coordinate
        return torch.cat([torch.sin(pxy), torch.cos(pxy),
                          torch.sin(pt),  torch.cos(pt)], dim=1)

    def forward(self, h):
        """Predict normalised temperature u from normalised input h."""
        return self.net(self.features(h))

    def alpha(self):
        """Return the physical diffusivity alpha in m^2/s."""
        return torch.exp(self.log_alpha)


## 4. Physics model

This section defines the heat-equation residual and boundary conditions. PyTorch autograd gives the derivatives needed for the PINN loss.

In [ ]:
def _derivs(model, h):
    """Return u and the derivatives needed for the heat equation.

    h contains normalised coordinates [xhat, yhat, that]. The returned
    derivatives are also with respect to these normalised coordinates.
    """
    h = h.requires_grad_(True)   # autograd must track h
    u = model(h)      # predicted normalised temperature

    # First derivatives du/dxhat, du/dyhat, du/dthat
    grad_outputs = torch.ones_like(u)
    grad_u = torch.autograd.grad(u, h, grad_outputs, create_graph=True)[0]
    u_xh, u_yh, u_th = grad_u[:,0:1], grad_u[:,1:2], grad_u[:,2:3]

    # Second derivatives for diffusion d2u/dxhat2 and d2u/dyhat2
    u_xx = torch.autograd.grad(u_xh, h, torch.ones_like(u_xh), create_graph=True)[0][:,0:1]
    u_yy = torch.autograd.grad(u_yh, h, torch.ones_like(u_yh), create_graph=True)[0][:,1:2]
    return u, u_xh, u_yh, u_th, u_xx, u_yy

def pde_residual(model, alpha, h):
    """Heat-equation residual at interior points h."""
    # Chain-rule factors Lx, Ly and T_MAX convert derivatives back to physical units
    _, _, _, u_th, u_xx, u_yy = _derivs(model, h)
    diffusion_term = u_xx/Lx**2 + u_yy/Ly**2
    return u_th.squeeze() - alpha*T_MAX*diffusion_term.squeeze()

def left_bc_u(y, t):
    """Known normalised temperature on the heated left boundary x=0."""
    # Time ramp: (1-exp(-t/tau)); spatial Gaussian: strongest at y=Ly/2
    return (1.0 - torch.exp(-t/TAU)) * torch.exp(-((y-Ly/2)/SIGMA)**2)

def robin_bc_residual(model, alpha, side, coord, t):
    """Robin boundary residual on the right, top, or bottom side.

    side chooses the boundary. coord is y for the right side, and x for
    top/bottom. t is the time vector for those boundary points.
    """
    # Build physical coordinates for the selected boundary
    if side == "right":
        x, y = torch.full_like(coord, Lx), coord
        sign, length = +1.0, Lx
    elif side == "top":
        x, y = coord, torch.full_like(coord, Ly)
        sign, length = +1.0, Ly
    elif side == "bottom":
        x, y = coord, torch.zeros_like(coord)
        sign, length = -1.0, Ly
    else:
        raise ValueError("side must be right/top/bottom")

    # Get temperature and boundary-normal derivative from the PINN
    u, u_xh, u_yh, _, _, _ = _derivs(model, to_hat(x, y, t))
    dudn_hat = u_xh.squeeze() if side == "right" else u_yh.squeeze()
    dTdn = dT * sign * dudn_hat / length   # convert du/dnhat to dT/dn
    temperature_K = T0 + dT * u.squeeze()

    # Energy balance: conductive heat leaving the plate should match heat lost to air/radiation
    k = alpha * RHO * CP
    conductive_flux_out = -k * dTdn
    heat_loss_out = H_CONV*(temperature_K-T0) + EPS*SIGMA_SB*(temperature_K**4 - T0**4)

    # Normalise the flux residual so it has a similar numerical size to other losses
    flux_scale = torch.abs(k.detach())*dT/length + H_CONV*dT + 4*EPS*SIGMA_SB*(T0+dT)**3*dT + 1e-12
    return (conductive_flux_out - heat_loss_out) / flux_scale


## 5. Training loop

Training uses five loss terms: data, PDE residual, initial condition, left heating boundary, and Robin convection/radiation boundaries. Stage 1 freezes $\alpha$; Stage 2 releases it.

### Sampling and validation helpers

These functions choose random sensor/PDE points during training and compute validation RMSE on held-out sensors.


In [ ]:
def sample_times(n, dev, lo, hi):
    """Return n random physical times between lo and hi.

    n = number of samples, dev = CPU/GPU device, lo/hi = min/max value.

    Some samples are intentionally placed near lo. In this problem, the
    early heating transient is important, so the model should see it often.
    """
    n_early = int(n * EARLY_FRAC)                  # early-time samples
    n_uniform = n - n_early                        # normal uniform samples

    uniform_times = lo + (hi - lo) * torch.rand(n_uniform, device=dev)
    early_times = lo + (hi - lo) * (torch.rand(n_early, device=dev) ** EARLY_POW)

    return torch.cat([uniform_times, early_times])

def sample_time_indices(n, Nt, dev):
    """Return n random row numbers for the COMSOL time table.

    Nt means number of time rows in the COMSOL table.

    This is the same idea as sample_times(), but here we need integer indices
    because sensor measurements are stored in rows of the CSV file.
    """
    n_early = int(n * EARLY_FRAC)
    n_uniform = n - n_early

    uniform_idx = torch.randint(0, Nt, (n_uniform,), device=dev)
    early_idx = (torch.rand(n_early, device=dev) ** EARLY_POW * (Nt - 1)).long()

    return torch.cat([uniform_idx, early_idx])

def validation_rmse_K(model, time_values, sensor_x, sensor_y, u_data, val_sensor_idx, dev):
    """Estimate validation RMSE in Kelvin on sensors not used for training.

    RMSE means root mean squared error. idx means index/id in the data table.
    val_sensor_idx contains the held-out sensor ids used only for validation.

    The function picks random validation sensor/time pairs, predicts the
    temperature there, and compares it with the COMSOL value.
    """
    time_idx = torch.randint(0, time_values.numel(), (VAL_BATCH,), device=dev)        
    val_positions = torch.randint(0, val_sensor_idx.numel(), (VAL_BATCH,), device=dev)
    sensor_idx = val_sensor_idx[val_positions]                                      

    h_val = to_hat(sensor_x[sensor_idx], sensor_y[sensor_idx], time_values[time_idx])
    u_pred = model(h_val).squeeze()

    error_K = (u_pred - u_data[time_idx, sensor_idx]) * dT
    return float(torch.sqrt(torch.mean(error_K**2)))


### Training loop

The loop builds each loss term, adds them together, and updates the network. At the start of Stage 2, alpha is unfrozen.


In [ ]:
def train(stage1=STAGE1, stage2=STAGE2, verbose=True):
    """Train the PINN and return the model, history, and data split.

    stage1: epochs with alpha frozen, mainly fitting the temperature field.
    stage2: epochs with alpha trainable, identifying the material parameter.
    """
    set_seed(SEED)
    t_np, x_np, y_np, T_np = load_comsol_sensor_data()
    Nt, Ns = T_np.shape   # Nt=time rows, Ns=sensor columns
   

    # Sensor numbers are 0, 1, ..., Ns-1. Shuffle once, then hold out 20%
    sensor_indices = np.arange(Ns)
    rng = np.random.default_rng(SEED)                                    
    rng.shuffle(sensor_indices)
    num_val = Ns // 5                                                     
    val_s = sensor_indices[:num_val]     # validation sensor ids
    train_s = sensor_indices[num_val:]   # training sensor ids

    # Move the data to the same device as the model
    time_values = torch.tensor(t_np, device=DEVICE)
    sensor_x = torch.tensor(x_np, device=DEVICE)
    sensor_y = torch.tensor(y_np, device=DEVICE)
    u_data = torch.tensor((T_np - T0) / dT, device=DEVICE)  # normalised COMSOL temperatures
    train_sensor_idx = torch.tensor(train_s, device=DEVICE)
    val_sensor_idx = torch.tensor(val_s, device=DEVICE)

    def sample_uniform(n, lo, hi):
        """Uniform random physical coordinates between lo and hi.

        lo/hi mean lower/upper limits, for example 0...Lx or 0...Ly.
        """
        return lo + (hi - lo) * torch.rand(n, device=DEVICE)

    model = PINN().to(DEVICE)
    model.log_alpha.requires_grad_(False) # alpha is frozen during stage 1

    # updating theta (network weights) and beta (log_alpha)
    opt = torch.optim.Adam([
        {"params": model.net.parameters(), "lr": 1e-3},
        {"params": [model.log_alpha], "lr": ALPHA_LR},
    ])

    # training history
    # History is only logged every LOG_EVERY epochs to keep the table small
    hist = {k: [] for k in ["epoch", "loss", "L_data", "L_pde", "L_bc", "alpha", "rmse_val"]}
    t0 = time.time()

    for ep in range(1, stage1 + stage2 + 1):                              
        if ep == stage1 + 1:
            model.log_alpha.requires_grad_(True)                         
            if verbose: print(f"--- stage 2 (identify alpha) begins at epoch {ep} ---")
        opt.zero_grad()

        # DATA LOSS: random mini-batch of measured COMSOL sensor/time points
        time_idx = sample_time_indices(DATA_BATCH, Nt, DEVICE)            
        train_positions = torch.randint(0, len(train_s), (DATA_BATCH,), device=DEVICE)
        sensor_idx = train_sensor_idx[train_positions]                    
        h_data = to_hat(sensor_x[sensor_idx], sensor_y[sensor_idx], time_values[time_idx])
        u_pred = model(h_data).squeeze()
        u_true = u_data[time_idx, sensor_idx]
        L_data = torch.mean((u_pred - u_true) ** 2)

        # INITIAL CONDITION LOSS: at t=0 the whole plate starts at T0, so u=0
        xi, yi = sample_uniform(BC_BATCH, 0, Lx), sample_uniform(BC_BATCH, 0, Ly)
        t_zero = torch.zeros(BC_BATCH, device=DEVICE)
        L_ic = torch.mean(model(to_hat(xi, yi, t_zero)) ** 2)

        # LEFT BOUNDARY LOSS: at x=0, match the known Gaussian heating function
        y_left = sample_uniform(BC_BATCH, 0, Ly)
        t_left = sample_times(BC_BATCH, DEVICE, 0.0, T_MAX)
        x_left = torch.zeros(BC_BATCH, device=DEVICE)
        u_left_pred = model(to_hat(x_left, y_left, t_left)).squeeze()
        u_left_true = left_bc_u(y_left, t_left)
        L_left = torch.mean((u_left_pred - u_left_true) ** 2)

        # PDE LOSS
        x_pde = sample_uniform(PDE_BATCH, 0, Lx)
        y_pde = sample_uniform(PDE_BATCH, 0, Ly)
        t_pde = sample_times(PDE_BATCH, DEVICE, T_PDE_MIN, T_MAX)
        L_pde = torch.mean(pde_residual(model, model.alpha(), to_hat(x_pde, y_pde, t_pde)) ** 2)

        # ROBIN BOUNDARY LOSS: random points on right/top/bottom boundaries
        y_right = sample_uniform(BC_BATCH, 0, Ly)                   # right side uses y as free coordinate
        t_right = sample_times(BC_BATCH, DEVICE, 0.0, T_MAX)
        x_top = sample_uniform(BC_BATCH, 0, Lx)                     # top side uses x as free coordinate
        t_top = sample_times(BC_BATCH, DEVICE, 0.0, T_MAX)
        x_bottom = sample_uniform(BC_BATCH, 0, Lx)                  # bottom side also uses x
        t_bottom = sample_times(BC_BATCH, DEVICE, 0.0, T_MAX)

        bc = torch.cat([
            robin_bc_residual(model, model.alpha(), "right", y_right, t_right),
            robin_bc_residual(model, model.alpha(), "top", x_top, t_top),
            robin_bc_residual(model, model.alpha(), "bottom", x_bottom, t_bottom),
        ])
        L_bc = torch.mean(bc ** 2)

        # One scalar objective for backpropagation
        loss = W_DATA * L_data + L_ic + L_left + W_PDE * L_pde + W_BC * L_bc
        loss.backward()
        opt.step()

        if ep % LOG_EVERY == 0 or ep == 1:
            with torch.no_grad():
                rmse = validation_rmse_K(model, time_values, sensor_x, sensor_y, u_data, val_sensor_idx, DEVICE)
            a = float(model.alpha().detach())
            hist["epoch"].append(ep)
            hist["loss"].append(float(loss.detach()))
            hist["L_data"].append(float(L_data.detach()))
            hist["L_pde"].append(float(L_pde.detach()))
            hist["L_bc"].append(float(L_bc.detach()))
            hist["alpha"].append(a)
            hist["rmse_val"].append(rmse)
            if verbose and (ep % 1000 == 0 or ep == 1 or ep == stage1 + 200):
                tag = "fit " if ep <= stage1 else "idfy"
                print(f"[{tag}] ep {ep:5d} | loss {float(loss.detach()):.2e} | alpha {a:.3e} "
                      f"({abs(a-ALPHA_TRUE)/ALPHA_TRUE*100:4.1f}%) | val RMSE {rmse:.3f} K")

    a = float(model.alpha().detach())
    print(f"\nDONE in {time.time()-t0:.0f}s | alpha = {a:.4e} "
          f"({abs(a-ALPHA_TRUE)/ALPHA_TRUE*100:.1f}% error) | val RMSE {hist['rmse_val'][-1]:.3f} K")
    return model, hist, (t_np, x_np, y_np, T_np, train_s, val_s)


## 6. Run training and save results

This cell trains the model, saves the history, writes final metrics, and stores a checkpoint for later plotting or inspection.

In [ ]:
model, hist, data = train()

# Save the training history
pd.DataFrame(hist).to_csv(os.path.join(OUT_DIR,"history.csv"), index=False)

a = float(model.alpha().detach())


with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"alpha = {a:.6e} m^2/s\n")
    f.write(f"alpha_error_pct = {abs(a-ALPHA_TRUE)/ALPHA_TRUE*100:.3f}\n")
    f.write(f"val_rmse_K = {hist['rmse_val'][-1]:.6f}\n")
    f.write("boundary_model = left Dirichlet heating + right/top/bottom convection+radiation\n")

# checkpoint.pt stores model weights
torch.save({"model_state": model.state_dict(), "alpha": a}, os.path.join(OUT_DIR, "checkpoint.pt"))

## 7. Main figures

The report uses four main figures: training diagnostics, sensor time traces, centreline heat penetration, and full-plate temperature contours. These plots are post-processing only; they do not affect training.

### Plot helpers

These helper lines prepare the history table and one prediction function used by the three figures.


In [ ]:
# Unpack the trained model data for plotting.
t_np, x_np, y_np, T_np, train_s, val_s = data
H = pd.DataFrame(hist)                                                     # H = history table for plotting
epochs = H["epoch"].to_numpy()
a_final = float(model.alpha().detach())

@torch.no_grad()
def predict_delta_T(x, y, t):
    """Predict temperature rise Delta T [K] at physical coordinates.

    x, y, t can be NumPy arrays of the same shape. The function returns a
    NumPy array with the same shape, in Kelvin above T0.
    """
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    t = np.asarray(t, dtype=np.float32)

    delta_T = np.empty(x.shape, dtype=np.float32)
    batch_size = 200_000                                                   # avoids memory spikes on large grids

    for start in range(0, x.size, batch_size):
        end = start + batch_size
        x_batch = torch.tensor(x[start:end], device=DEVICE, dtype=torch.float32)
        y_batch = torch.tensor(y[start:end], device=DEVICE, dtype=torch.float32)
        t_batch = torch.tensor(t[start:end], device=DEVICE, dtype=torch.float32)
        u_batch = model(to_hat(x_batch, y_batch, t_batch)).squeeze()        # normalised PINN prediction
        delta_T[start:end] = dT * u_batch.cpu().numpy()
    return delta_T

def closest_sensor(x_target, y_target):
    """Return the index of the exported sensor closest to a requested point."""
    distance_squared = (x_np - x_target)**2 + (y_np - y_target)**2
    return int(np.argmin(distance_squared))


### Figure 0: COMSOL sensor layout and measured snapshots

This figure uses only the exported COMSOL sensor CSV. Each marker is one virtual sensor; no interpolation is used. It is useful for explaining where the training/validation data come from.


In [ ]:
def format_plate_axis(axis):
    """Apply the same plate outline, labels, and grid to sensor plots."""
    axis.set_xlim(-0.004, Lx + 0.004)
    axis.set_ylim(-0.004, Ly + 0.004)
    axis.set_aspect("equal")
    axis.set_xlabel("x [m]")
    axis.set_ylabel("y [m]")
    axis.set_xticks([0.00, 0.05, 0.10])
    axis.set_yticks([0.00, 0.025, 0.05])
    axis.add_patch(plt.Rectangle((0, 0), Lx, Ly, fill=False, lw=1.3, edgecolor="black"))
    axis.plot([0, 0], [0, Ly], color="#d73027", lw=3.0)                  # heated left edge
    axis.grid(alpha=0.18, lw=0.6)

# Separate sensor-location figure: training sensors and held-out validation sensors.
fig, ax = plt.subplots(figsize=(9, 4.8))
format_plate_axis(ax)
ax.scatter(x_np[train_s], y_np[train_s], s=42, facecolor="#334155", edgecolor="white",
           linewidth=0.7, label=f"training sensors ({len(train_s)})", zorder=3)
ax.scatter(x_np[val_s], y_np[val_s], s=58, facecolor="#f59e0b", edgecolor="black",
           linewidth=0.6, label=f"validation sensors ({len(val_s)})", zorder=4)
ax.set_title("COMSOL virtual sensor locations")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "sensor_locations.png"), dpi=180)
plt.show()
plt.close(fig)

# Sensor temperatures at four times. This uses measured COMSOL values only.
snapshot_times = [5.0, 20.0, 100.0, 300.0]
delta_T_data = T_np - T0                                                   # convert Kelvin to temperature rise
vmax = float(np.nanmax(delta_T_data))                                      # shared colour scale for all snapshots

fig, axes = plt.subplots(2, 2, figsize=(10.5, 5.9), constrained_layout=True)
axes = axes.ravel()
scatter = None
for axis, requested_time in zip(axes, snapshot_times):
    time_id = int(np.argmin(np.abs(t_np - requested_time)))                # closest exported COMSOL time row
    actual_time = float(t_np[time_id])
    format_plate_axis(axis)
    scatter = axis.scatter(x_np, y_np, c=delta_T_data[time_id], s=54, cmap="inferno",
                           vmin=0, vmax=vmax, edgecolor="white", linewidth=0.45, zorder=3)
    axis.set_title(f"COMSOL sensors, t = {actual_time:g} s")
fig.colorbar(scatter, ax=axes.tolist(), shrink=0.88, label=r"$\Delta T$ [K]")
fig.suptitle("Measured temperature rise at the virtual sensors", fontsize=15)
fig.savefig(os.path.join(OUT_DIR, "sensor_temperature_snapshots.png"), dpi=180)
plt.show()
plt.close(fig)

# Combined version for slides: layout + temperature snapshots in one image.
fig = plt.figure(figsize=(13.33, 7.5))
grid = fig.add_gridspec(2, 3, width_ratios=[1.15, 1, 1], wspace=0.28, hspace=0.32)

ax_layout = fig.add_subplot(grid[:, 0])
format_plate_axis(ax_layout)
ax_layout.scatter(x_np[train_s], y_np[train_s], s=38, facecolor="#334155", edgecolor="white",
                  linewidth=0.7, label=f"train ({len(train_s)})", zorder=3)
ax_layout.scatter(x_np[val_s], y_np[val_s], s=54, facecolor="#f59e0b", edgecolor="black",
                  linewidth=0.6, label=f"validation ({len(val_s)})", zorder=4)
ax_layout.set_title("sensor layout")
ax_layout.legend(loc="lower right", fontsize=9)

snapshot_axes = [fig.add_subplot(grid[i, j]) for i in range(2) for j in [1, 2]]
scatter = None
for axis, requested_time in zip(snapshot_axes, snapshot_times):
    time_id = int(np.argmin(np.abs(t_np - requested_time)))
    actual_time = float(t_np[time_id])
    format_plate_axis(axis)
    scatter = axis.scatter(x_np, y_np, c=delta_T_data[time_id], s=42, cmap="inferno",
                           vmin=0, vmax=vmax, edgecolor="white", linewidth=0.4, zorder=3)
    axis.set_title(f"t = {actual_time:g} s", fontsize=11)
fig.colorbar(scatter, ax=snapshot_axes, shrink=0.82, label=r"$\Delta T$ [K]")
fig.suptitle("COMSOL virtual sensors: position and measured temperature rise", fontsize=18, y=0.98)
fig.text(0.035, 0.035, "No interpolation: each marker is one exported COMSOL sensor.",
         fontsize=11, color="#475569")
fig.savefig(os.path.join(OUT_DIR, "sensor_layout_and_snapshots.png"), dpi=180)
plt.show()
plt.close(fig)


### Figure 1: training diagnostics

This figure checks whether the loss decreases, whether validation RMSE is reasonable, and how the estimated alpha moves during training.


In [ ]:
# Training dashboard: shows whether alpha, losses, and validation RMSE behave sensibly.
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

# Alpha is plotted in units of 1e-5 so the numbers are easy to read.
ax[0, 0].plot(epochs, H["alpha"].to_numpy() * 1e5, color="tab:blue", linewidth=2)
ax[0, 0].axhline(ALPHA_TRUE * 1e5, color="black", linestyle="--")
ax[0, 0].set_title("(a) alpha estimate")
ax[0, 0].set_xlabel("epoch")
ax[0, 0].set_ylabel(r"alpha [$10^{-5}$ m$^2$/s]")
ax[0, 0].grid(alpha=0.3)

alpha_error = np.abs(H["alpha"].to_numpy() - ALPHA_TRUE) / ALPHA_TRUE * 100
ax[0, 1].semilogy(epochs, alpha_error, color="tab:blue", linewidth=2)
ax[0, 1].axhline(5, color="green", linestyle="--", label="5%")
ax[0, 1].axhline(10, color="orange", linestyle="--", label="10%")
ax[0, 1].set_title("(b) alpha error")
ax[0, 1].set_xlabel("epoch")
ax[0, 1].set_ylabel("error [%]")
ax[0, 1].legend()
ax[0, 1].grid(alpha=0.3)

# semilogy is useful because the loss terms can differ by orders of magnitude.
for loss_name in ["loss", "L_data", "L_pde", "L_bc"]:
    ax[1, 0].semilogy(epochs, H[loss_name], label=loss_name)
ax[1, 0].set_title("(c) training losses")
ax[1, 0].set_xlabel("epoch")
ax[1, 0].set_ylabel("loss")
ax[1, 0].legend()
ax[1, 0].grid(alpha=0.3)

# Validation RMSE is measured on held-out sensors and converted back to Kelvin.
ax[1, 1].semilogy(epochs, H["rmse_val"], color="tab:purple", linewidth=2)
ax[1, 1].set_title("(d) validation RMSE")
ax[1, 1].set_xlabel("epoch")
ax[1, 1].set_ylabel("RMSE [K]")
ax[1, 1].grid(alpha=0.3)

fig.suptitle(f"Training diagnostics | alpha = {a_final:.3e}")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/training_dashboard.png", dpi=150)
plt.show()


### Figure 2: sensor time traces

Solid lines are COMSOL sensor data. Dashed lines are PINN predictions at the same sensor locations.


In [ ]:
# Four representative sensors: near the heated side, middle, far side, and upper-right area.
sensor_targets = [
    (0.0115, 0.023),
    (0.0505, 0.023),
    (0.0700, 0.023),
    (0.0895, 0.0455),
]
sensor_ids = [closest_sensor(x0, y0) for x0, y0 in sensor_targets]
colors = plt.cm.viridis(np.linspace(0, 0.85, len(sensor_ids)))

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(13, 5))

for sensor_id, color in zip(sensor_ids, colors):
    # Build one time trace at a fixed sensor position.
    x_sensor = np.full(t_np.shape, x_np[sensor_id], dtype=np.float32)
    y_sensor = np.full(t_np.shape, y_np[sensor_id], dtype=np.float32)
    pinn_delta_T = predict_delta_T(x_sensor, y_sensor, t_np)
    comsol_delta_T = T_np[:, sensor_id] - T0
    label = f"({x_np[sensor_id]:.3f}, {y_np[sensor_id]:.3f})"

    ax_full.plot(t_np, comsol_delta_T, color=color, linewidth=2.2, label=label)
    ax_full.plot(t_np, pinn_delta_T, "--", color=color, linewidth=1.5)
    ax_zoom.plot(t_np, comsol_delta_T, color=color, linewidth=2.2)
    ax_zoom.plot(t_np, pinn_delta_T, "--", color=color, linewidth=1.5)

ax_full.set_xlim(0, T_MAX)
ax_full.set_title("full 0-300 s")
ax_full.legend(title="sensor (x,y) [m]", fontsize=8)

ax_zoom.set_xlim(0, 30)
ax_zoom.set_title("early-time zoom (0-30 s)")

for axis in (ax_full, ax_zoom):
    axis.set_xlabel("t [s]")
    axis.set_ylabel(r"$\Delta T$ [K]")
    axis.grid(alpha=0.3)

fig.suptitle("Sensor temperatures: COMSOL (solid) vs PINN (dashed)")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/sensor_fit.png", dpi=150)
plt.show()


### Figure 3: centreline profile

This plot shows how heat penetrates from the heated left edge along the middle line of the plate.


In [ ]:
# Centreline means y approximately equal to Ly/2, across the plate from left to right.
y_mid = Ly / 2.0
centerline_sensors = np.where(np.abs(y_np - y_mid) < 0.004)[0]
centerline_sensors = centerline_sensors[np.argsort(x_np[centerline_sensors])]
x_line = np.linspace(0, Lx, 200, dtype=np.float32)                         # smooth PINN line
plot_times = [5.0, 20.0, 100.0, 300.0]
colors = plt.cm.plasma(np.linspace(0, 0.85, len(plot_times)))

fig, ax = plt.subplots(figsize=(8, 5))

for plot_time, color in zip(plot_times, colors):
    time_id = int(np.argmin(np.abs(t_np - plot_time)))
    time_value = float(t_np[time_id])
    t_line = np.full(x_line.shape, time_value, dtype=np.float32)
    y_line = np.full(x_line.shape, y_mid, dtype=np.float32)
    pinn_delta_T = predict_delta_T(x_line, y_line, t_line)
    comsol_delta_T = T_np[time_id, centerline_sensors] - T0

    ax.plot(x_line, pinn_delta_T, color=color, linewidth=2, label=f"t = {time_value:g} s")
    ax.scatter(
        x_np[centerline_sensors],
        comsol_delta_T,
        color=color,
        s=28,
        edgecolors="black",
        linewidths=0.3,
        zorder=5,
    )

ax.set_xlabel("x [m]  (0 = heated left edge)")
ax.set_ylabel(r"$\Delta T$ [K]")
ax.set_title("Centreline profile at y = Ly/2: PINN (line) vs COMSOL sensors (dots)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/centerline_profile.png", dpi=150)
plt.show()



### Figure 4: temperature contours over time

This is the same type of plot as the contour image used in the presentation. It is made from the trained PINN, not from a new COMSOL run.

It shows the predicted temperature rise over the full plate at several fixed times. This is mainly a qualitative visual check: heat should start near the heated left boundary and spread into the plate over time.

In [ ]:
# Put the model in evaluation mode before plotting. This does not change the
# trained weights; it only tells PyTorch that we are no longer training.
model.eval()
os.makedirs(OUT_DIR, exist_ok=True)

contour_times = [5.0, 20.0, 100.0, 300.0]
x_grid = np.linspace(0, Lx, 180, dtype=np.float32)                         # regular x grid for plotting
y_grid = np.linspace(0, Ly, 90, dtype=np.float32)                          # regular y grid for plotting
X, Y = np.meshgrid(x_grid, y_grid)                                         # 2D plate grid
contour_levels = np.linspace(0.0, dT, 41)                                  # fixed shared scale: 0...30 K

def contour_grid_at_time(time_value):
    """Evaluate the trained PINN on the full x-y grid at one fixed time."""
    t_grid = np.full(X.size, time_value, dtype=np.float32)
    delta_T_grid = predict_delta_T(X.ravel(), Y.ravel(), t_grid)
    # Clip only for visualisation: the plotted quantity is temperature rise.
    return np.clip(delta_T_grid.reshape(X.shape), 0.0, dT)

# One combined figure for the report/presentation.
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()

for axis, plot_time in zip(axes, contour_times):
    Z = contour_grid_at_time(plot_time)
    cf = axis.contourf(X, Y, Z, levels=contour_levels, cmap="viridis")     # cf = filled contour object
    axis.set_title(f"t = {plot_time:g} s")
    axis.set_aspect("equal")
    axis.set_xlabel("x [m]")
    axis.set_ylabel("y [m]")

cbar = fig.colorbar(cf, ax=axes.tolist(), shrink=0.92)
cbar.set_label(r"$\Delta T$ [K]")
fig.suptitle("PINN temperature contours over time")
fig.savefig(os.path.join(OUT_DIR, "temperature_contours_over_time.png"), dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

# Also save separate contour images, like the older notebook versions did.
for plot_time in contour_times:
    Z = contour_grid_at_time(plot_time)
    fig, axis = plt.subplots(figsize=(10, 4), constrained_layout=True)
    cf = axis.contourf(X, Y, Z, levels=contour_levels, cmap="viridis")     # cf = filled contour object
    fig.colorbar(cf, ax=axis, label=r"$\Delta T$ [K]")
    axis.set_title(f"PINN temperature rise at t = {plot_time:g} s")
    axis.set_xlabel("x [m]")
    axis.set_ylabel("y [m]")
    axis.set_aspect("equal")
    fig.savefig(os.path.join(OUT_DIR, f"contour_t{plot_time:g}.png"), dpi=200)
    plt.show()
    plt.close(fig)

print("Saved contour plots to", OUT_DIR)


## 8. Saved repeated-run statistics

The main model above is the final Fourier-feature PINN. The repeated runs below are not a new method: they are the same training setup saved from several runs with different starting values. The notebook reads the saved CSV files and makes the comparison plots here, so the analysis can be explained from one Jupyter file.


### 8.1 Fourier model: sensitivity to the starting $\alpha$

This table contains repeated runs of the final Fourier-feature model. Only the initial guess for $\alpha$ changes; the training settings are kept fixed. This is used to show whether the inverse estimate is robust or still depends on the starting point.


In [ ]:
from pathlib import Path
from IPython.display import display

# This CSV contains repeated Fourier-model runs with different starting alpha values.
alpha_stats_path = Path("outputs_alpha_sensitivity/alpha_sensitivity_summary.csv")
if not alpha_stats_path.exists():
    raise FileNotFoundError(
        "Missing outputs_alpha_sensitivity/alpha_sensitivity_summary.csv. "
        "Use the saved repeated-run results or rerun the alpha-start experiments."
    )

alpha_stats = pd.read_csv(alpha_stats_path)

# Keep only the columns needed for the report discussion.
alpha_stats_view = alpha_stats[[
    "ALPHA_INIT", "alpha", "alpha_error_pct", "val_rmse_K", "final_L_data", "final_L_pde", "final_L_bc"
]].copy()

# Multiply by 1e5 so alpha values are easier to read in a table.
alpha_stats_view["ALPHA_INIT_x1e5"] = alpha_stats_view["ALPHA_INIT"] * 1e5
alpha_stats_view["alpha_x1e5"] = alpha_stats_view["alpha"] * 1e5
alpha_stats_view = alpha_stats_view[[
    "ALPHA_INIT_x1e5", "alpha_x1e5", "alpha_error_pct", "val_rmse_K",
    "final_L_data", "final_L_pde", "final_L_bc"
]]

# Rounded view for the report/defence discussion.
display(alpha_stats_view.round({
    "ALPHA_INIT_x1e5": 2,
    "alpha_x1e5": 3,
    "alpha_error_pct": 2,
    "val_rmse_K": 3,
    "final_L_data": 6,
    "final_L_pde": 6,
    "final_L_bc": 6,
}))


In [ ]:
# Plot the saved Fourier-model repeated runs directly from the CSV table.
# This figure shows whether alpha recovery depends on the initial alpha guess.
alpha_stats_sorted = alpha_stats.sort_values("ALPHA_INIT")
x_init = alpha_stats_sorted["ALPHA_INIT"].to_numpy() * 1e5                # initial alpha guesses
y_alpha = alpha_stats_sorted["alpha"].to_numpy() * 1e5                   # recovered alpha values
y_err = alpha_stats_sorted["alpha_error_pct"].to_numpy()                 # parameter error [%]
y_rmse = alpha_stats_sorted["val_rmse_K"].to_numpy()                     # temperature fit error [K]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(x_init, y_alpha, "o-", linewidth=2, color="tab:blue")
axes[0].axhline(ALPHA_TRUE * 1e5, color="black", linestyle="--", label="true alpha")
axes[0].set_xlabel(r"initial alpha [$10^{-5}$ m$^2$/s]")
axes[0].set_ylabel(r"recovered alpha [$10^{-5}$ m$^2$/s]")
axes[0].set_title("Fourier PINN: recovered alpha")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(x_init, y_err, "o-", linewidth=2, color="tab:orange", label="alpha error")
axes[1].axhline(5, color="green", linestyle="--", linewidth=1, label="5%")
axes[1].axhline(10, color="orange", linestyle="--", linewidth=1, label="10%")
axes[1].set_xlabel(r"initial alpha [$10^{-5}$ m$^2$/s]")
axes[1].set_ylabel("alpha error [%]")
axes[1].grid(alpha=0.3)

# Use a second y-axis because RMSE [K] and alpha error [%] have different units.
rmse_axis = axes[1].twinx()
rmse_axis.plot(x_init, y_rmse, "s--", color="tab:purple", label="validation RMSE")
rmse_axis.set_ylabel("validation RMSE [K]")

lines, labels = axes[1].get_legend_handles_labels()                     # legend items from left axis
lines2, labels2 = rmse_axis.get_legend_handles_labels()                  # legend items from right axis
axes[1].legend(lines + lines2, labels + labels2, loc="upper right")
axes[1].set_title("Parameter error and temperature fit")

fig.suptitle("Saved repeated runs of the final Fourier-feature PINN")
fig.savefig(os.path.join(OUT_DIR, "notebook_alpha_sensitivity_summary.png"), dpi=150)
plt.show()
plt.close(fig)


### 8.2 Architecture comparison from saved runs

This table compares three network choices that were tested earlier: a plain MLP, a bigger MLP, and the final Fourier-feature MLP. The comparison is read from a saved CSV table and plotted here. It supports the modelling choice: Fourier features improve the temperature fit, but the spread in recovered $\alpha$ remains, which points to weak identifiability.


In [ ]:
comparison_path = Path("outputs_comparison/comparison_table.csv")
if not comparison_path.exists():
    raise FileNotFoundError(
        "Missing outputs_comparison/comparison_table.csv. "
        "Keep the saved architecture-comparison table or rerun those comparison experiments."
    )

# This table was produced from earlier repeated runs of three model choices.
comparison_table = pd.read_csv(comparison_path)

# One summary row per architecture: temperature fit and alpha recovery spread.
comparison_summary = comparison_table.groupby("variant").agg(
    mean_rmse_K=("val_rmse_K", "mean"),
    std_rmse_K=("val_rmse_K", "std"),
    mean_alpha_error_pct=("alpha_error_pct", "mean"),
    best_alpha_error_pct=("alpha_error_pct", "min"),
    min_alpha=("alpha", "min"),
    max_alpha=("alpha", "max"),
).reset_index()
comparison_summary["min_alpha_x1e5"] = comparison_summary["min_alpha"] * 1e5
comparison_summary["max_alpha_x1e5"] = comparison_summary["max_alpha"] * 1e5

summary_view = comparison_summary[[
    "variant", "mean_rmse_K", "std_rmse_K", "mean_alpha_error_pct",
    "best_alpha_error_pct", "min_alpha_x1e5", "max_alpha_x1e5"
]]
display(summary_view.round({
    "mean_rmse_K": 3,
    "std_rmse_K": 3,
    "mean_alpha_error_pct": 1,
    "best_alpha_error_pct": 1,
    "min_alpha_x1e5": 2,
    "max_alpha_x1e5": 2,
}))


In [ ]:
# Plot the architecture comparison directly from the saved CSV table.
variant_order = ["baseline (plain MLP)", "bigger MLP", "Fourier features"]
colors = {
    "baseline (plain MLP)": "#d62728",
    "bigger MLP": "#ff7f0e",
    "Fourier features": "#2ca02c",
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)

# Left plot: every dot is one run from a different starting alpha.
for i, variant in enumerate(variant_order):
    subset = comparison_table[comparison_table["variant"] == variant]
    axes[0].scatter(
        [i] * len(subset),
        subset["alpha"].to_numpy() * 1e5,
        s=70,
        color=colors[variant],
        edgecolors="black",
        zorder=3,
    )
axes[0].axhline(ALPHA_TRUE * 1e5, color="black", linestyle="--", label="true alpha")
axes[0].set_xticks(range(len(variant_order)))
axes[0].set_xticklabels(variant_order, rotation=15, ha="right")
axes[0].set_ylabel(r"recovered alpha [$10^{-5}$ m$^2$/s]")
axes[0].set_title("Recovered alpha across starts")
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

# Right plot: average validation RMSE for each architecture.
means = []
stds = []
for variant in variant_order:
    subset = comparison_table[comparison_table["variant"] == variant]
    means.append(subset["val_rmse_K"].mean())
    stds.append(subset["val_rmse_K"].std())

axes[1].bar(
    range(len(variant_order)),
    means,
    yerr=stds,
    capsize=5,
    color=[colors[v] for v in variant_order],
    edgecolor="black",
    linewidth=0.7,
)
axes[1].set_xticks(range(len(variant_order)))
axes[1].set_xticklabels(variant_order, rotation=15, ha="right")
axes[1].set_ylabel("validation RMSE [K]")
axes[1].set_title("Temperature fit: mean +/- std")
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Saved architecture comparison runs")
fig.savefig(os.path.join(OUT_DIR, "notebook_architecture_comparison.png"), dpi=150)
plt.show()
plt.close(fig)


## 9. Some references



- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, Journal of Computational Physics, 2019. https://doi.org/10.1016/j.jcp.2018.10.045
- Özisik & Orlande, *Inverse Heat Transfer: Fundamentals and Applications*. https://www.taylorfrancis.com/books/mono/10.1201/9780203749784/inverse-heat-transfer-necat-ozisik
- Beck, Blackwell & St. Clair, *Inverse Heat Conduction: Ill-Posed Problems*. https://www.osti.gov/biblio/5792952
- Cai et al., *Physics-Informed Neural Networks for Heat Transfer Problems*, Journal of Heat Transfer, 2021. https://doi.org/10.1115/1.4050542
- Oommen & Srinivasan, *Solving Inverse Heat Transfer Problems Without Surrogate Models*, JCISE, 2022. https://doi.org/10.1115/1.4053800
- Fourier features project page and code: https://bmild.github.io/fourfeat/
- PyTorch autograd: https://docs.pytorch.org/docs/stable/autograd.html
- Matplotlib `contourf`, used for the temperature contour maps: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.contourf.html
